# Day 4

## Tokenizing (tách token) bằng code

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [ ]:
tokens

In [ ]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

In [ ]:
encoding.decode([326])

# Và một chủ đề nữa!

### Ảo giác về "memory" (bộ nhớ)

Nhiều bạn có lẽ đã biết rồi. Nhưng với ai chưa biết — đây có thể là khoảnh khắc "AHA" (bừng tỉnh)!

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

### Bạn nên rất thoải mái với những gì ô tiếp theo đang làm!

_Tôi đang tạo một instance (thể hiện) mới của OpenAI Python Client library (thư viện client Python của OpenAI), một wrapper (lớp bọc) nhẹ quanh việc gọi HTTP tới một endpoint (điểm cuối API) để gọi GPT LLM, hoặc các nhà cung cấp LLM khác_

In [ ]:
from openai import OpenAI

openai = OpenAI()

### Một message (tin nhắn) gửi tới OpenAI là một list (danh sách) các dicts (từ điển)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
    ]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

### OK, bây giờ hãy hỏi một câu follow-up (câu hỏi tiếp theo)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

### Khoan, cái gì??

Chúng ta vừa nói với bạn mà!

Chuyện gì đang xảy ra??

Đây là điều quan trọng: mỗi lần gọi một LLM đều hoàn toàn STATELESS (không lưu trạng thái). Mỗi lần đều là một lời gọi hoàn toàn mới. Với tư cách AI engineers (kỹ sư AI), NHIỆM VỤ CỦA CHÚNG TA là nghĩ ra các kỹ thuật để tạo ấn tượng rằng LLM có "memory" (bộ nhớ).

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

## Tóm lại

Xin lỗi nếu điều này đã quá hiển nhiên với bạn — nhưng vẫn nên củng cố:

1. Mỗi lần gọi LLM đều stateless (không lưu trạng thái)
2. Mỗi lần chúng ta đều truyền toàn bộ cuộc hội thoại cho đến hiện tại vào input prompt (prompt đầu vào)
3. Việc này tạo ảo giác rằng LLM có memory (bộ nhớ) — nó dường như giữ được context (ngữ cảnh) của cuộc hội thoại
4. Nhưng đó là một trick (mẹo); nó chỉ là hệ quả của việc cung cấp toàn bộ cuộc hội thoại, mỗi lần
5. Một LLM chỉ dự đoán các token (đơn vị văn bản) tiếp theo có khả năng cao nhất trong chuỗi; nếu chuỗi đó chứa "My name is Ed" rồi sau đó "What's my name?" thì nó sẽ dự đoán... Ed!

Sản phẩm ChatGPT dùng đúng trick này — mỗi lần bạn gửi một message (tin nhắn), toàn bộ cuộc hội thoại được truyền vào.

"Vậy có nghĩa là mỗi lần chúng ta phải trả thêm tiền cho toàn bộ cuộc hội thoại đến nay?"

Đúng vậy. Và đó là điều chúng ta MUỐN. Chúng ta muốn LLM dự đoán các token tiếp theo trong chuỗi, nhìn lại toàn bộ cuộc hội thoại. Chúng ta muốn phép tính compute (tính toán) đó xảy ra, nên cần trả hóa đơn điện cho nó!

